# TP05: Estructuración y Agregación
## Laboratorio (Herramientas) - Universidad del Aconcagua
### Unidad 3: Herramientas en la Nube de Modelado de Datos

---

### 🎯 Objetivos del Trabajo Práctico

1. Guardar dataframes como **tablas estructuradas**
2. Aplicar **funciones de agregación** sobre métricas comerciales
3. Crear **vistas** y esquemas lógicos
4. Realizar **agrupamientos complejos**
5. Transformar grupos con funciones personalizadas

---

### 📁 Caso de Estudio: Tablas de Ventas Agregadas

Crearemos tablas estructuradas con métricas agregadas de la panadería.

### 🕰️ Duración Estimada: 2 horas

In [0]:
# Configurar rutas del proyecto
import os
from pathlib import Path

# Obtener usuario actual automáticamente desde Spark
# Esto hace que el notebook sea 100% portable entre usuarios
USUARIO = spark.sql("SELECT current_user()").collect()[0][0]
BASE_USER = Path(f"/Workspace/Users/{USUARIO}")
BASE_LABORATORIO = BASE_USER / "Laboratorio"
BASE_DATASETS = BASE_LABORATORIO / "05 - Datasets"

print(f"📍 Usuario: {USUARIO}")
print(f"\n📂 Rutas configuradas:")
print(f"  Usuario:     {BASE_USER}")
print(f"  Laboratorio: {BASE_LABORATORIO}")
print(f"  Datasets:    {BASE_DATASETS}")

# Verificar que las rutas existen
if BASE_DATASETS.exists():
    archivos_csv = len([f for f in os.listdir(BASE_DATASETS) if f.endswith('.csv')])
    print(f"\n✅ Rutas verificadas correctamente")
    print(f"   Archivos CSV encontrados: {archivos_csv}")
else:
    print(f"\n⚠️ Advertencia: La carpeta de datasets no existe")

In [0]:
# Importar librerías necesarias
import pandas as pd  # Manipulación y análisis de datos tabulares
import numpy as np   # Operaciones numéricas y arrays
from pyspark.sql import SparkSession  # Motor distribuido de procesamiento
from pyspark.sql import functions as F  # Funciones SQL de Spark (sum, avg, count, etc.)
from pyspark.sql.window import Window  # Funciones de ventana (ranking, acumulados)

# Inicializar Spark Session
# SparkSession es el punto de entrada para trabajar con Spark SQL y DataFrames
# builder.getOrCreate() obtiene la sesión existente o crea una nueva
spark = SparkSession.builder.getOrCreate()

print("✅ Librerías importadas")
print(f"   Spark version: {spark.version}")
print(f"\n💡 Listo para trabajar con DataFrames distribuidos")

## Parte 1: Carga de Datos como Spark DataFrames

### 📂 De Pandas a Spark

Vamos a cargar los CSV con Pandas y convertirlos a Spark DataFrames para trabajar con tablas estructuradas.

In [0]:
# Cargar datos con Pandas usando las rutas configuradas
# BASE_DATASETS fue definida en la celda anterior con pathlib
# Estos DataFrames de Pandas luego se convertirán a Spark DataFrames

# pd.read_csv() lee archivos CSV y los convierte en DataFrames de Pandas
df_productos_pd = pd.read_csv(BASE_DATASETS / 'productos.csv')
df_sucursales_pd = pd.read_csv(BASE_DATASETS / 'sucursales.csv')
df_clientes_pd = pd.read_csv(BASE_DATASETS / 'clientes.csv')
# parse_dates=['fecha'] convierte automáticamente la columna 'fecha' a tipo datetime
df_ventas_pd = pd.read_csv(BASE_DATASETS / 'ventas.csv', parse_dates=['fecha'])
df_detalles_pd = pd.read_csv(BASE_DATASETS / 'detalles_ventas.csv')

print("✅ Datos cargados con Pandas")
print(f"  Productos: {len(df_productos_pd):,} registros")
print(f"  Sucursales: {len(df_sucursales_pd):,} registros")
print(f"  Clientes: {len(df_clientes_pd):,} registros")
print(f"  Ventas: {len(df_ventas_pd):,} registros")
print(f"  Detalles: {len(df_detalles_pd):,} registros")

In [0]:
# Convertir Pandas DataFrames a Spark DataFrames
# spark.createDataFrame() convierte un DataFrame de Pandas a Spark
# Esto permite:
# - Procesamiento distribuido en clúster
# - Manejo de datasets que no caben en memoria
# - Uso de SQL y transformaciones optimizadas de Spark

df_productos = spark.createDataFrame(df_productos_pd)
df_sucursales = spark.createDataFrame(df_sucursales_pd)
df_clientes = spark.createDataFrame(df_clientes_pd)
df_ventas = spark.createDataFrame(df_ventas_pd)
df_detalles = spark.createDataFrame(df_detalles_pd)

print("✅ DataFrames convertidos a Spark")
print("\n📄 Schema de productos (estructura de columnas y tipos):")
df_productos.printSchema()

print("\n💡 Ahora estos DataFrames pueden procesarse de forma distribuida")

## Parte 2: Guardar como Tablas Delta

### 💾 Persistencia estructurada

Las tablas Delta permiten almacenar datos de forma eficiente con transacciones ACID, versionado y optimizaciones automáticas.

In [0]:
# Guardar DataFrames como tablas Delta en el catálogo
# Las tablas Delta son el formato optimizado de Databricks que ofrece:
# - Transacciones ACID (atomicidad, consistencia, aislamiento, durabilidad)
# - Versionado automático (time travel)
# - Schema evolution (cambios de esquema sin perder datos)
# - Optimizaciones automáticas (compactación, indexación)

print("💾 Creando tablas Delta...\n")

# .write.mode('overwrite') reemplaza la tabla si ya existe
# .saveAsTable() guarda el DataFrame como tabla en el catálogo de Databricks

# Tabla de productos (catálogo)
df_productos.write.mode('overwrite').saveAsTable('panaderia_productos')
print("✅ Tabla panaderia_productos creada")

# Tabla de sucursales (ubicaciones)
df_sucursales.write.mode('overwrite').saveAsTable('panaderia_sucursales')
print("✅ Tabla panaderia_sucursales creada")

# Tabla de clientes
df_clientes.write.mode('overwrite').saveAsTable('panaderia_clientes')
print("✅ Tabla panaderia_clientes creada")

# Tabla de ventas PARTICIONADA
# .partitionBy('anio', 'mes') divide la tabla en subdirectorios por año y mes
# Beneficio: Las consultas que filtran por fecha son muchísimo más rápidas
# porque solo leen las particiones relevantes (partition pruning)
df_ventas.write.mode('overwrite') \
    .partitionBy('anio', 'mes') \
    .saveAsTable('panaderia_ventas')
print("✅ Tabla panaderia_ventas creada (particionada por anio/mes)")
print("   💡 Las consultas por fecha serán más rápidas gracias al particionamiento")

# Tabla de detalles de ventas (granularidad: producto por venta)
df_detalles.write.mode('overwrite').saveAsTable('panaderia_detalles_ventas')
print("✅ Tabla panaderia_detalles_ventas creada")

print("\n✅ Todas las tablas Delta creadas exitosamente")
print("   Ahora pueden consultarse con SQL o PySpark desde cualquier notebook")

In [0]:
%sql
-- Ver las tablas creadas
SHOW TABLES LIKE 'panaderia_*'

## Parte 3: Agregaciones Complejas con Spark SQL

### 📈 Métricas de negocio agregadas

Vamos a crear tablas agregadas que resuman las ventas desde diferentes perspectivas.

In [0]:
# Leer las tablas Delta recién creadas desde el catálogo
# spark.table() carga una tabla registrada en el catálogo como DataFrame
ventas = spark.table('panaderia_ventas')
detalles = spark.table('panaderia_detalles_ventas')
productos = spark.table('panaderia_productos')
sucursales = spark.table('panaderia_sucursales')

# Crear tabla agregada: ventas por sucursal y mes
# Este patrón es fundamental para análisis de series temporales por ubicación

# .groupBy() agrupa registros por las columnas especificadas
# .agg() aplica funciones de agregación sobre cada grupo
ventas_sucursal_mes = ventas.groupBy('sucursal_id', 'anio', 'mes').agg(
    F.count('venta_id').alias('cantidad_ventas'),      # Número de transacciones
    F.sum('total').alias('facturacion_total'),         # Suma total de ventas
    F.avg('total').alias('ticket_promedio'),           # Promedio por transacción
    F.min('total').alias('venta_minima'),              # Venta más pequeña
    F.max('total').alias('venta_maxima')               # Venta más grande
).orderBy('anio', 'mes', 'sucursal_id')  # Ordenar cronológicamente

# Guardar como tabla Delta para consultas futuras
ventas_sucursal_mes.write.mode('overwrite').saveAsTable('panaderia_ventas_sucursal_mes')

print("✅ Tabla de agregación creada: panaderia_ventas_sucursal_mes")
print("   💡 Uso: Análisis de tendencias y comparación entre sucursales")
print("\n📄 Primeras filas (vista previa):")
ventas_sucursal_mes.show(10)

In [0]:
# Unir detalles con productos para obtener nombres y categorías
# .join() funciona como JOIN en SQL
# Tipo 'left': mantiene todos los registros de detalles, incluso si no hay match en productos
ventas_productos = detalles.join(productos, 'producto_id', 'left')

# Agregar por producto (análisis de rendimiento de catálogo)
# Este tipo de agregación es clave para:
# - Identificar productos estrella (best sellers)
# - Detectar productos de bajo rendimiento
# - Optimizar inventario y estrategia de precios

ventas_por_producto = ventas_productos.groupBy(
    'producto_id', 'nombre', 'categoria'
).agg(
    F.sum('cantidad').alias('unidades_vendidas'),      # Total de unidades
    F.sum('subtotal').alias('facturacion'),            # Ingresos totales
    F.count('venta_id').alias('numero_ventas'),        # Frecuencia de compra
    F.avg(detalles['precio_unitario']).alias('precio_promedio')  # Precio promedio de venta
).orderBy(F.desc('facturacion'))  # Ordenar por facturación descendente

# Guardar como tabla Delta
ventas_por_producto.write.mode('overwrite').saveAsTable('panaderia_ventas_por_producto')

print("✅ Tabla de agregación creada: panaderia_ventas_por_producto")
print("   💡 Uso: Análisis de rendimiento de productos y optimización de catálogo")
print("\n🏆 Top 10 productos por facturación:")
ventas_por_producto.show(10, truncate=False)

In [0]:
%sql
-- Crear tabla agregada por categoría usando SQL puro
CREATE OR REPLACE TABLE panaderia_ventas_por_categoria AS
SELECT 
  p.categoria,
  COUNT(DISTINCT d.venta_id) as numero_ventas,
  SUM(d.cantidad) as unidades_vendidas,
  SUM(d.subtotal) as facturacion_total,
  AVG(d.precio_unitario) as precio_promedio,
  ROUND(AVG(d.descuento_porcentaje), 2) as descuento_promedio
FROM panaderia_detalles_ventas d
JOIN panaderia_productos p ON d.producto_id = p.producto_id
GROUP BY p.categoria
ORDER BY facturacion_total DESC;

SELECT * FROM panaderia_ventas_por_categoria;

## Parte 4: Funciones de Ventana (Window Functions)

### 🕹️ Cálculos por grupos con contexto

Las window functions permiten calcular rankings, acumulados y comparaciones dentro de particiones de datos.

In [0]:
# Calcular ranking de productos más vendidos por sucursal
# Este análisis usa WINDOW FUNCTIONS (funciones de ventana)
# que permiten cálculos por grupos sin perder filas individuales

from pyspark.sql.window import Window

# Unir ventas con detalles y productos
# Múltiples joins para tener toda la información en un solo DataFrame
ventas_completas = ventas.join(detalles, 'venta_id').join(productos, 'producto_id')

# Agrupar por sucursal y producto
# Calculamos métricas agregadas para cada combinación sucursal-producto
ventas_sucursal_producto = ventas_completas.groupBy(
    'sucursal_id', 'producto_id', 'nombre', 'categoria'
).agg(
    F.sum('cantidad').alias('unidades_vendidas'),
    F.sum('subtotal').alias('facturacion')
)

# Definir la especificación de ventana (window spec)
# partitionBy('sucursal_id'): Crear una "ventana" separada para cada sucursal
# orderBy(F.desc('facturacion')): Dentro de cada ventana, ordenar por facturación
# Esto permite calcular el ranking DENTRO de cada sucursal
window_spec = Window.partitionBy('sucursal_id').orderBy(F.desc('facturacion'))

# Calcular ranking usando row_number() sobre la ventana
# row_number(): Asigna un número secuencial (1, 2, 3...) a cada fila en su ventana
# El producto con mayor facturación en cada sucursal recibe ranking=1
ranking_productos = ventas_sucursal_producto.withColumn(
    'ranking', F.row_number().over(window_spec)
).filter(F.col('ranking') <= 5)  # Filtrar solo el Top 5 por sucursal

print("🏆 Top 5 productos por sucursal:")
print("   💡 Cada sucursal tiene su propio ranking (gracias a partitionBy)")
ranking_productos.orderBy('sucursal_id', 'ranking').show(15, truncate=False)

In [0]:
# Calcular ventas acumuladas por mes
# Los acumulados son fundamentales para tracking de objetivos y tendencias

# Primero: Agregar ventas por mes
ventas_mensuales = ventas.groupBy('anio', 'mes').agg(
    F.sum('total').alias('facturacion_mensual')
).orderBy('anio', 'mes')  # IMPORTANTE: Ordenar cronológicamente

# Definir ventana para cálculo acumulado
# orderBy('anio', 'mes'): Procesar filas en orden cronológico
# rowsBetween(unboundedPreceding, currentRow): Desde el inicio hasta la fila actual
# Esto significa: "sumar desde el primer mes hasta el mes actual"
window_acum = Window.orderBy('anio', 'mes').rowsBetween(
    Window.unboundedPreceding,  # Desde el inicio
    Window.currentRow            # Hasta la fila actual
)

# Calcular métricas con funciones de ventana
ventas_acumuladas = ventas_mensuales.withColumn(
    # Acumulado: suma de todos los meses hasta ahora
    'facturacion_acumulada', 
    F.sum('facturacion_mensual').over(window_acum)
).withColumn(
    # Crecimiento vs mes anterior
    # lag('columna'): Obtiene el valor de la fila anterior en la ventana
    # (valor_actual - valor_anterior) / valor_anterior * 100 = % de crecimiento
    'crecimiento_vs_anterior', 
    F.round(
        (F.col('facturacion_mensual') - 
         F.lag('facturacion_mensual').over(Window.orderBy('anio', 'mes'))) / 
        F.lag('facturacion_mensual').over(Window.orderBy('anio', 'mes')) * 100, 
        2
    )
)

print("📈 Ventas mensuales con acumulado y crecimiento:")
print("   💡 'facturacion_acumulada' muestra el total YTD (Year-To-Date)")
print("   💡 'crecimiento_vs_anterior' muestra la variación mensual (%)")
ventas_acumuladas.show(24)

## Parte 5: Ejercicios Prácticos

### ✍️ Ejercicios para Resolver

#### **Ejercicio 1**: Tabla agregada de clientes VIP
Crea una tabla que agregue las ventas de clientes VIP, mostrando cuánto ha gastado cada cliente VIP en total.

In [0]:
# EJERCICIO 1: Ventas de clientes VIP
# Pista: Une ventas con clientes, filtra por es_vip=True, agrupa y suma

# Tu código aquí:
clientes = spark.table('panaderia_clientes')

ventas_vip = ventas.join(
    clientes.filter(F.col('es_vip') == True),
    'cliente_id',
    'inner'
).groupBy('cliente_id', 'nombre').agg(
    F.count('venta_id').alias('numero_compras'),
    F.sum('total').alias('total_gastado'),
    F.avg('total').alias('ticket_promedio')
).orderBy(F.desc('total_gastado'))

print("🌟 CLIENTES VIP - TOP 10 POR GASTO")
ventas_vip.show(10, truncate=False)

#### **Ejercicio 2**: Participación de mercado por categoría
Calcula el porcentaje de facturación que representa cada categoría sobre el total.

In [0]:
%sql
-- EJERCICIO 2: Participación de mercado
-- Pista: Usa SUM() OVER() para calcular el total general

SELECT 
  p.categoria,
  SUM(d.subtotal) as facturacion,
  ROUND(SUM(d.subtotal) * 100.0 / SUM(SUM(d.subtotal)) OVER(), 2) as porcentaje_participacion
FROM panaderia_detalles_ventas d
JOIN panaderia_productos p ON d.producto_id = p.producto_id
GROUP BY p.categoria
ORDER BY facturacion DESC

## 🎯 Resumen del TP05

### ✅ Qué aprendimos:

1. **Configuración portable**: Rutas dinámicas con USUARIO y pathlib para notebooks reproducibles
2. **Tablas Delta**: Guardamos DataFrames como tablas persistentes con transacciones ACID
3. **Particionamiento**: Optimizamos consultas particionando por año/mes
4. **Agregaciones complejas**: Calculamos métricas agrupadas con groupBy y agg
5. **SQL en Spark**: Usamos tanto Python como SQL para crear tablas agregadas
6. **Window Functions**: Aplicamos rankings, acumulados y comparaciones con ventanas
7. **Transformaciones avanzadas**: Combinamos joins, agregaciones y funciones de ventana
8. **Agregaciones geoespaciales con H3**: Análisis por zona geográfica (Parte Bonus al final)

### 📊 Tablas creadas:

* `panaderia_productos` - Catálogo de productos
* `panaderia_ventas` - Transacciones (particionada por año/mes)
* `panaderia_ventas_sucursal_mes` - Agregado por sucursal y mes
* `panaderia_ventas_por_producto` - Agregado por producto
* `panaderia_ventas_por_categoria` - Agregado por categoría

### 🗺️ Parte Bonus:

Al final del notebook encontrarás:
* Agregaciones por zona H3 (hexágonos geoespaciales)
* Window functions espaciales
* Cambio de resolución H3 (zoom in/out)
* Joins espaciales por proximidad

### 🚀 Próximos pasos:

En el **TP06** aprenderemos a:
* Crear features para machine learning
* Aplicar encoding a variables categóricas
* Escalar y normalizar variables numéricas
* Preparar datasets para modelos predictivos

---

**📝 Excelente trabajo! Ahora sabes cómo estructurar y agregar datos a escala con Spark.**

In [0]:
print("=" * 80)
print("AGREGACIÓN 1: MÉTRICAS POR ZONA H3")
print("=" * 80)

# Unir ventas con clientes para obtener h3_index
# Solo consideramos ventas con cliente identificado (cliente_id no es null)
ventas_geo = df_ventas_pd[df_ventas_pd['cliente_id'].notna()].merge(
    df_clientes_pd[['cliente_id', 'h3_index']], 
    on='cliente_id'
)

print(f"\nVentas georeferenciadas: {len(ventas_geo):,} ({len(ventas_geo)/len(df_ventas_pd)*100:.1f}%)")

# GROUP BY por zona H3
# Cada hexágono H3 representa una pequeña área geográfica (~100m² en resolución 9)
metricas_zona = ventas_geo.groupby('h3_index').agg({
    'total': ['sum', 'mean', 'count'],  # Facturación total, promedio, número de ventas
    'venta_id': 'nunique',               # Ventas únicas
    'cliente_id': 'nunique'              # Clientes únicos
}).round(2)

# Renombrar columnas para claridad
metricas_zona.columns = [
    'facturacion_total', 
    'ticket_promedio', 
    'num_transacciones',
    'ventas_unicas',
    'clientes_unicos'
]

metricas_zona = metricas_zona.reset_index()

# Agregar ranking de zonas por facturación
metricas_zona['ranking_facturacion'] = metricas_zona['facturacion_total'].rank(ascending=False)

print(f"\n📊 Estadísticas por Zona H3:")
print(f"   Total de zonas: {len(metricas_zona)}")
print(f"   Facturación promedio por zona: ${metricas_zona['facturacion_total'].mean():,.0f}")
print(f"   Ticket promedio general: ${metricas_zona['ticket_promedio'].mean():,.0f}")

print(f"\n🏆 Top 10 Zonas por Facturación:")
print(metricas_zona.nlargest(10, 'facturacion_total')[[
    'h3_index', 'facturacion_total', 'ticket_promedio', 'clientes_unicos'
]])

In [0]:
print("\n" + "=" * 80)
print("AGREGACIÓN 2: WINDOW FUNCTIONS ESPACIALES")
print("=" * 80)

# Calcular diferencia vs. promedio general
# Permite identificar zonas que están por encima o por debajo del promedio
metricas_zona['diff_vs_promedio'] = (
    metricas_zona['ticket_promedio'] - metricas_zona['ticket_promedio'].mean()
)

# Calcular porcentaje de diferencia
metricas_zona['pct_vs_promedio'] = (
    (metricas_zona['ticket_promedio'] / metricas_zona['ticket_promedio'].mean() - 1) * 100
).round(2)

# Clasificar zonas según su desempeño
def clasificar_zona(pct):
    """Clasifica una zona según su desempeño vs. promedio"""
    if pct > 20:
        return '🔥 Premium'
    elif pct > 5:
        return '🟢 Alta'
    elif pct > -5:
        return '🟡 Media'
    elif pct > -20:
        return '🟠 Baja'
    else:
        return '🔵 Muy Baja'

metricas_zona['categoria_zona'] = metricas_zona['pct_vs_promedio'].apply(clasificar_zona)

print(f"\n📈 Distribución de Zonas por Categoría:")
print(metricas_zona['categoria_zona'].value_counts().sort_index())

print(f"\n📊 Zonas Premium (>20% sobre promedio):")
zonas_premium = metricas_zona[metricas_zona['categoria_zona'] == '🔥 Premium'].sort_values(
    'pct_vs_promedio', ascending=False
)
if len(zonas_premium) > 0:
    print(zonas_premium[[
        'h3_index', 'ticket_promedio', 'pct_vs_promedio', 'clientes_unicos'
    ]].head())
    print(f"\n💡 Insight: {len(zonas_premium)} zonas premium representan oportunidad de expansión")
else:
    print("   No hay zonas premium en el dataset")

In [0]:
print("\n" + "=" * 80)
print("AGREGACIÓN 3: MÉTRICAS POR CATEGORÍA Y ZONA")
print("=" * 80)

# Unir detalles con productos y ventas para obtener datos completos
# Este patrón de múltiples joins es común en análisis dimensional
ventas_completas = df_detalles.merge(
    df_productos[['producto_id', 'categoria']], 
    on='producto_id'
).merge(
    df_ventas[['venta_id', 'cliente_id']], 
    on='venta_id'
).merge(
    df_clientes[['cliente_id', 'h3_index']], 
    on='cliente_id',
    how='left'  # Left join para mantener ventas sin cliente identificado
)

# Filtrar solo ventas georeferenciadas
ventas_completas = ventas_completas[ventas_completas['h3_index'].notna()]

# Agregar por zona y categoría (análisis bidimensional)
categoria_zona = ventas_completas.groupby(['h3_index', 'categoria']).agg({
    'subtotal': 'sum',   # Ventas totales
    'cantidad': 'sum'    # Unidades vendidas
}).reset_index()

categoria_zona.columns = ['h3_index', 'categoria', 'ventas', 'unidades']

print(f"\n📊 Métricas por Categoría y Zona:")
print(f"   Combinaciones únicas: {len(categoria_zona)}")

# Producto más vendido por zona (categoría dominante)
producto_top_zona = ventas_completas.groupby(['h3_index', 'categoria']).agg({
    'subtotal': 'sum'
}).reset_index()

# Ordenar y obtener el top 1 por zona
producto_top_zona = producto_top_zona.sort_values(
    ['h3_index', 'subtotal'], ascending=[True, False]
).groupby('h3_index').first().reset_index()

print(f"\n🎯 Categoría Dominante por Zona (Top 10):")
print(producto_top_zona.nlargest(10, 'subtotal')[['h3_index', 'categoria', 'subtotal']])

# Resumen por categoría (todas las zonas)
print(f"\n🔍 Resumen de Ventas por Categoría (todas las zonas):")
resumen_cat = categoria_zona.groupby('categoria').agg({
    'ventas': 'sum',
    'unidades': 'sum'
}).sort_values('ventas', ascending=False)
print(resumen_cat)

In [0]:
print("\n" + "=" * 80)
print("AGREGACIÓN 4: ZOOM OUT - CAMBIO DE RESOLUCIÓN H3")
print("=" * 80)

print("\n🔍 Concepto: Cambiar resolución para ver patrones más amplios")
print("   Resolución 9 (actual): ~0.1 km² por hexágono")
print("   Resolución 7 (padre): ~5 km² por hexágono (zoom out)")

# Convertir a resolución 7 (barrios/zonas más amplias)
# h3.cell_to_parent() sube en la jerarquía H3
# Cada hexágono de resolución 7 contiene múltiples hexágonos de resolución 9
metricas_zona['h3_res7'] = metricas_zona['h3_index'].apply(
    lambda x: h3.cell_to_parent(x, 7)
)

# Agregar a resolución 7 (sumar todas las zonas pequeñas dentro de cada barrio)
metricas_barrio = metricas_zona.groupby('h3_res7').agg({
    'facturacion_total': 'sum',      # Sumar facturación de todas las zonas
    'num_transacciones': 'sum',      # Sumar transacciones
    'clientes_unicos': 'sum',        # Sumar clientes (puede haber duplicados)
    'h3_index': 'count'              # Contar cuántas zonas pequeñas hay en este barrio
}).reset_index()

metricas_barrio.columns = [
    'h3_barrio', 'facturacion', 'transacciones', 'clientes', 'num_zonas_pequenas'
]

# Calcular facturación promedio por zona pequeña dentro del barrio
metricas_barrio['facturacion_por_zona'] = (
    metricas_barrio['facturacion'] / metricas_barrio['num_zonas_pequenas']
).round(2)

print(f"\n📊 Visión de Barrios (resolución 7):")
print(f"   Total de barrios: {len(metricas_barrio)}")
print(f"   Zonas pequeñas por barrio: {metricas_barrio['num_zonas_pequenas'].mean():.1f}")

print(f"\n🏆 Top 5 Barrios por Facturación:")
print(metricas_barrio.nlargest(5, 'facturacion')[[
    'h3_barrio', 'facturacion', 'clientes', 'facturacion_por_zona'
]])

print(f"\n💡 Uso práctico:")
print("   - Resolución 7: Estrategia de marketing regional")
print("   - Resolución 9: Optimización de rutas de delivery")
print("   - Cambiar resolución permite análisis a diferentes escalas geográficas")

### 📝 Ejercicio: Joins Espaciales con H3

**Objetivo**: Unir datos por proximidad geográfica.

**Tareas**:

1. **Join por distancia**:
   - Para cada cliente, encontrar la sucursal más cercana
   - Usar `h3.grid_distance()` para calcular distancia en hexágonos
   - Agregar columna `sucursal_mas_cercana` al dataset de clientes

2. **Agregar por proximidad a sucursal**:
   - Calcular facturación promedio de clientes según distancia a sucursal
   - Grupos: 0-2 hex (muy cerca), 3-5 hex (cerca), 6+ hex (lejos)
   - ¿Los clientes más cercanos compran más?

3. **Vecindarios**:
   - Usar `h3.grid_disk(h3_index, radio)` para obtener hexágonos vecinos
   - Calcular para cada zona: facturación total en vecindario (radio 2)
   - Identificar zonas "rodeadas" de alta facturación

**Pista**: `h3.grid_distance(h3_a, h3_b)` devuelve número de hexágonos entre dos puntos.

---

## 🗺️ Parte Bonus: Agregaciones Geoespaciales con H3

### Por qué Agregar por Zona Geográfica

Las agregaciones espaciales permiten:
- 📍 **Identificar patrones geográficos** en datos de negocio
- 📈 **Analizar performance por zona** (facturación, clientes, productos)
- 🎯 **Optimizar estrategias** localizadas (marketing, logística, apertura de sucursales)
- 🛡️ **Detectar anomalías espaciales** (zonas con bajo rendimiento)

### Operaciones Comunes

1. **GROUP BY h3_index**: Agregar métricas por zona
2. **Window Functions espaciales**: Comparar zona vs. promedio general
3. **Joins espaciales**: Unir datos por proximidad geográfica
4. **Agregaciones jerárquicas**: Cambiar resolución H3 para zoom out/in

---

In [0]:
# Importar librerías para análisis geoespacial
import pandas as pd  # Manipulación de datos
import numpy as np   # Operaciones numéricas
import h3           # Sistema de indexación geoespacial jerárquico de Uber

# Cargar datasets usando las rutas configuradas
# BASE_DATASETS ya fue definida en la celda de configuración
df_clientes = pd.read_csv(BASE_DATASETS / 'clientes.csv')
df_ventas = pd.read_csv(BASE_DATASETS / 'ventas.csv')
df_productos = pd.read_csv(BASE_DATASETS / 'productos.csv')
df_detalles = pd.read_csv(BASE_DATASETS / 'detalles_ventas.csv')
df_sucursales = pd.read_csv(BASE_DATASETS / 'sucursales.csv')

print("✅ Datasets cargados para análisis geoespacial")
print(f"   Total clientes: {len(df_clientes):,}")
print(f"   Clientes georeferenciados (con H3): {df_clientes['h3_index'].notna().sum():,}")
print(f"   Cobertura geoespacial: {df_clientes['h3_index'].notna().sum()/len(df_clientes)*100:.1f}%")